# 课后作业：图上随机游走的块 I/O 优化（两阶段）

《计算机系统设计》（研究生一年级）· 学生版

**规则速览**（完整版见 `README.md`，评测以 `walk_sim.py` 输出为准，禁止修改骨架核心语义）：

- karate 图（34 点 / 78 边）分 **4 块**，内存最多驻留 **M=2** 块；每点发起 **i=10** 次、步长 **l=20** 的游走。
- 同步轮次：每轮调度器选驻留集 R*（≤M 块），可推进的 walker 各走 1 步。
  - 一阶可推进：`block(vid) ∈ R`；二阶可推进：`block(vid) ∈ R` 且 `block(pid) ∈ R`（step=0 仅需前者）。
- 加载计数：块从外存调入计 1 次（含首轮冷启动）；cache hit 不计；重入重复计。
- 自定义分区约束：**恰好 4 块，每块 4–17 点**。
- 统一种子 2026；除加载次数外须同时报告 `walk_updating_rate` 与 `io_utilization`；**不收敛的低加载次数无效**。

提交物：本 notebook（含运行结果）+ `mypart.txt` + ≤2 页报告（pdf/md）。

## 阶段一 · 任务 1：复现一阶 baseline（正确性 40 分）

运行下面 cell，应得到 `total_loads = 52`（顺序等分 + state-aware 调度）。若数字不符，检查目录与种子。

In [ ]:
import json, os
from walk_sim import run_sim

DATA = 'data'
s0, trace0, _ = run_sim(DATA, order=1)
print(json.dumps(s0, indent=2, ensure_ascii=False))
assert s0['total_loads'] == 52 and s0['converged'], 'baseline 未复现，请检查环境'

## 阶段一 · 任务 2：自定义分区（优化加分，L≤47 得 +20，L≤42 得 +40）

在 `my_partition()` 中实现你的分区策略（可用 networkx 的社区发现 / 谱方法 / 局部搜索等，也可手工设计）。

**思考提示**（不剧透）：① 跨块边少是否就够？② walker 在块内能连续走多少步才是关键；③ 别忘了约束：恰好 4 块、每块 4–17 点。

In [ ]:
import networkx as nx

G = nx.read_edgelist(os.path.join(DATA, 'karate.edgelist'), nodetype=int)

def my_partition():
    """返回 dict: vid -> block_id (0..3)。约束：恰好 4 块，每块 4-17 点。"""
    meta = {}
    # TODO: 在此实现你的分区策略
    # 示例（顺序等分，请替换）：
    nodes = sorted(G.nodes())
    sizes = [9, 9, 8, 8]
    idx = 0
    for b, sz in enumerate(sizes):
        for v in nodes[idx:idx + sz]:
            meta[v] = b
        idx += sz
    return meta

meta = my_partition()
assert set(meta.values()) == {0, 1, 2, 3}, '必须恰好 4 块'
sizes = [sum(1 for v in meta if meta[v] == b) for b in range(4)]
assert all(4 <= s <= 17 for s in sizes), f'块大小越界: {sizes}'

with open('mypart.txt', 'w') as f:
    for v in sorted(meta):
        f.write(f'{v} {meta[v]}\n')

s1, _, _ = run_sim(DATA, order=1, meta='mypart.txt')
print(json.dumps(s1, indent=2, ensure_ascii=False))
L0 = s0['total_loads']
print(f"相对 baseline: {s1['total_loads']/L0:.2%}  (达标线 90%={0.9*L0:.0f}, 优秀线 80%={0.8*L0:.0f})")

## 阶段二 · 任务 1：观察二阶 naive 的饥饿现象（正确性 25 分）

运行下面 cell，然后在报告中回答：

1. naive 为什么不收敛？用 SOWalker 的 **non-updatable walks** 概念解释，并指出 walk matrix 中哪些项被一阶式调度系统性忽略。
2. naive 的 `total_loads` 比 AUW 还低，为什么说这个数字没有意义？应该结合哪个指标判断？
3. 为什么二阶的加载次数系统性高于一阶？

In [ ]:
s_nv, _, _ = run_sim(DATA, order=2, policy='naive')
s_aw, _, _ = run_sim(DATA, order=2, policy='auw')
print('naive:', json.dumps(s_nv, ensure_ascii=False))
print('auw  :', json.dumps(s_aw, ensure_ascii=False))
assert not s_nv['converged'] and s_aw['converged']

## 阶段二 · 任务 2：自定义调度函数与/或分区，联合优化二阶（正确性 25 分 + 优化加分，L≤86 得 +15，L≤71 得 +30）

在 `my_scheduler()` 中实现你的二阶调度策略（函数签名与内置调度器一致；骨架强制 |R*|≤M 并计数）。

**思考提示**：① AUW 只数"可更新游走数"，能否进一步考虑块内**连续多步**的潜力？② 换出决策能否利用 walk matrix 的行/列和？③ 调度与分区是耦合的——为 AUW 结构量身设计的分区可能优于通用社区分区。

In [ ]:
from walk_sim import sched_auw

def my_scheduler(walkers, resident, M, store, order):
    """返回驻留集 R*（set，|R*| <= M）。
    可用信息：walkers（.vid .pid .step .done）、store.block_of[v]、store.blocks、store.adj[v]。
    平局请取块号较小者（确定性要求）。"""
    # TODO: 在 AUW 基础上改进，或完全自定义
    return sched_auw(walkers, resident, M, store, order)

# 可同时使用你在阶段一的分区：meta='mypart.txt'
s2, trace2, _ = run_sim(DATA, order=2, scheduler=my_scheduler)
print(json.dumps(s2, indent=2, ensure_ascii=False))
L2 = s_aw['total_loads']
if s2['converged']:
    print(f"相对 AUW baseline: {s2['total_loads']/L2:.2%}  (达标线 85%={0.85*L2:.0f}, 优秀线 70%={0.7*L2:.0f})")
else:
    print('未收敛：调度策略无法在规定时间内完成游走，请检查')

## 加分项（+10，任选）

```python
# 1) 极端 p/q 对块访问分布的影响
run_sim(DATA, order=2, policy='auw', p=0.25, q=4.0)   # 强回跳
run_sim(DATA, order=2, policy='auw', p=4.0, q=0.25)   # 类 DFS
# 2) 8 块 / M=2（驻留比 25%，更接近论文场景）：修改 config.json 或用自定义 meta 切 8 块
# 3) ER 对照图（无社区结构）：
run_sim(DATA, order=1, edgelist='data/er.edgelist', meta='data/er_meta.txt')
```

## 报告要求

- ≤2 页，含：方法（分区/调度策略的思想）、三指标结果表、与 baseline 的对比、权衡讨论、加载次数下界的论证。
- 评分：阶段一 = 正确性 40 + 提交完整 20 + 优化 ≤40；阶段二 = 正确性 50 + 提交完整 20 + 优化 ≤30。加分项 +10。封顶 100。